## Import Libraries

In [1]:
import sys
import os

sys.path.append(os.path.abspath('..'))

In [2]:
from dotenv import load_dotenv
from pathlib import Path
from sklearn.metrics import roc_curve
from qdrant_client import QdrantClient
from qdrant_client import models
from torch.utils.data import DataLoader, TensorDataset
from pytorch_metric_learning import losses
from utils.embedding_model import embedding_model
import numpy as np
import time
import torch
import psutil
from tqdm import tqdm
import wandb

## Setup Training Variables

In [3]:
load_dotenv(".env")
load_dotenv("../.env")
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
load_dotenv(".env")

model_name = 'embedding_v4'
ratio = '90:10'
train_split = '90'
seeder = os.getenv("SEED")
window_len = os.getenv("WINDOW_SIZE")
stride_len = os.getenv("STRIDE")
num_batch = os.getenv("BATCH_SIZE")
num_epoch = os.getenv("EPOCHS")
margin = 0.2
data_type = os.getenv("DATA_TYPE", "eo")  # Read from env, default to 'eo'
wandb_name = model_name + "_" + data_type + '_train_' + train_split + '_' + str(seeder) + '_' + str(window_len) + '_' + str(stride_len) + '_b' + str(num_batch) + '_e' + str(num_epoch) + '_margin_' + str(margin)
print(wandb_name)

embedding_v4_ec_train_90_2024_2_2_b128_e100_margin_0.2


In [4]:
load_dotenv(".env")
BASE_PATH = os.getenv("BASE_PATH")
PREPROCESSED_PATH = os.getenv("PREPROCESSED_PATH")
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

suffix = f"{os.getenv('WINDOW_SIZE').replace('.', '')}_{os.getenv('STRIDE').replace('.', '')}"
preprocessed_dir = Path(BASE_PATH + PREPROCESSED_PATH)

X_train = np.load(preprocessed_dir / f'X_{data_type}_train_{suffix}_seed{seeder}.npy')
y_train = np.load(preprocessed_dir / f'y_{data_type}_train_{suffix}_seed{seeder}.npy')
X_test = np.load(preprocessed_dir / f'X_{data_type}_test_{suffix}_seed{seeder}.npy')
y_test = np.load(preprocessed_dir / f'y_{data_type}_test_{suffix}_seed{seeder}.npy')

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)

X_train: (2943, 64, 320) y_train: (2943,)
X_test: (327, 64, 320) y_test: (327,)


In [5]:
# Raw cropped data is split/windowed in 01_data_preprocessing.ipynb.
# Training uses the saved train/val/test arrays loaded above.


In [6]:
wandb.login(key=os.getenv("WANDB_API_KEY"))
run = wandb.init(
    entity="chocomaltt",
    project="eeg-biometric-system",
    name=wandb_name,
    config={
        "model_name": model_name,
        "ratio": ratio,
        "data_type": data_type,
        "train_split": train_split,
        "seeder": seeder,
        "window_len": os.getenv("WINDOW_SIZE"),
        "stride_len": os.getenv("STRIDE"),
        "num_batch": os.getenv("BATCH_SIZE"),
        "epoch": os.getenv("EPOCHS")
    },
    tags=[model_name, 'train_' + str(train_split), str(seeder), str(window_len), str(stride_len), str(num_batch), str(num_epoch), str(margin)]
)

process = psutil.Process(os.getpid())
initial_memory = psutil.virtual_memory()
wandb.log({
    "resource/logging_check": 1,
    "resource/cpu_percent": psutil.cpu_percent(interval=1),
    "resource/process_cpu_percent": process.cpu_percent(interval=None),
    "resource/memory_percent": initial_memory.percent,
    "resource/memory_used_gb": initial_memory.used / (1024 ** 3),
    "resource/process_memory_gb": process.memory_info().rss / (1024 ** 3),
})

wandb: Loading settings from /home/chocomaltt/.config/wandb/settings


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.


wandb: [wandb.login()] Using explicit session credentials for http://localhost:8080.


wandb: Appending key for localhost:8080 to your netrc file: /home/chocomaltt/.netrc


wandb: Currently logged in as: chocomaltt to http://localhost:8080. Use `wandb login --relogin` to force relogin


wandb: Tracking run with wandb version 0.25.1


wandb: Run data is saved locally in /home/chocomaltt/Kuliah/eeg-biometric-system/wandb/run-20260531_132136-o970vor3
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run embedding_v4_ec_train_90_2024_2_2_b128_e100_margin_0.2


wandb: ⭐️ View project at http://localhost:8080/chocomaltt/eeg-biometric-system


wandb: 🚀 View run at http://localhost:8080/chocomaltt/eeg-biometric-system/runs/o970vor3


In [7]:
print("Loaded split arrays from preprocessing notebook.")
print("Train labels:", np.unique(y_train, return_counts=True))
print("Test labels:", np.unique(y_test, return_counts=True))


Loaded split arrays from preprocessing notebook.
Train labels: (array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
        78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
        91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
       104, 105, 106, 107, 108]), array([27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27,
       27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27,
       27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27,
       27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27, 27,
       27, 27, 27, 27, 27,

In [8]:
X_train_t = torch.from_numpy(X_train.copy()).float()
y_train_t = torch.from_numpy(y_train.copy()).long()

X_test_t = torch.from_numpy(X_test.copy()).float()
y_test_t = torch.from_numpy(y_test.copy()).long()

train_ds = TensorDataset(X_train_t, y_train_t)
test_ds = TensorDataset(X_test_t, y_test_t)

train_loader = DataLoader(
    train_ds,
    batch_size=int(os.getenv("BATCH_SIZE")),
    shuffle=True,
    num_workers=int(os.getenv("NUM_WORKERS")),
    drop_last=True
)
test_loader = DataLoader(
    test_ds,
    batch_size=int(os.getenv("BATCH_SIZE")),
    shuffle=False,
    num_workers=int(os.getenv("NUM_WORKERS")),
    drop_last=False
)

In [9]:
model = embedding_model()
model.to(os.getenv("DEVICE"))

embedding_model(
  (input): Sequential(
    (0): LazyConv2d(0, 64, kernel_size=(1, 1), stride=(1, 1), padding=same)
    (1): SELU()
  )
  (conv2_temporal): Sequential(
    (0): LazyConv2d(0, 32, kernel_size=(4, 4), stride=(1, 1), padding=same)
    (1): SELU()
  )
  (batch_normalization): LazyBatchNorm2d(0, eps=32, momentum=0.1, affine=True, track_running_stats=True)
  (elu): ELU(alpha=1.0)
  (MaxPool2d): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
  (conv2_spatial): Sequential(
    (0): LazyConv2d(0, 64, kernel_size=(2, 2), stride=(1, 1), padding=same)
    (1): SELU()
  )
  (lstm): LSTM(2048, 128, batch_first=True)
  (dense): Sequential(
    (0): LazyLinear(in_features=0, out_features=128, bias=True)
    (1): SELU()
  )
)

In [10]:
# Pastikan DEVICE sudah di-set (GPU kalau ada, kalau nggak CPU)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Lakukan "Dry Run" untuk membangunkan layer Lazy
with torch.no_grad():
    # Ambil 1 sampel saja dari X_train_t (Ingat, ECG sudah kita buang)
    sample_eeg = X_train_t[:1].to(DEVICE, non_blocking=True)

    sample_eeg = sample_eeg.unsqueeze(1)
    
    # Masukkan ke model. Setelah baris ini lewat, dimensi layer Lazy resmi terbentuk!
    _ = model(sample_eeg)

# 3. Hitung Parameter
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✓ Model berjalan di: {DEVICE}")
print(f"✓ Model initialized - Total params: {total_params:,}, Trainable: {trainable_params:,}")

# 4. Cek Memori GPU (Opsional)
if torch.cuda.is_available():
    print(f"GPU Memory: {torch.cuda.memory_allocated()/1e9:.2f}GB allocated")

✓ Model berjalan di: cuda
✓ Model initialized - Total params: 1,172,896, Trainable: 1,172,896
GPU Memory: 0.01GB allocated


/home/chocomaltt/Kuliah/eeg-biometric-system/eeg/lib/python3.10/site-packages/torch/nn/modules/conv.py:548: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1025.)
  return F.conv2d(


In [11]:
client = QdrantClient(url="http://localhost:6333")

if not client.collection_exists(wandb_name):
    client.create_collection(
        collection_name=wandb_name,
        vectors_config=models.VectorParams(size=128, distance=models.Distance.EUCLID),
    )

In [12]:
LEARNING_RATE = float(os.getenv("LEARNING_RATE", 1e-4))
EPOCHS = int(os.getenv("EPOCHS", 100))
torch.backends.cudnn.benchmark = True  # Set to True untuk performa maksimal, tapi pastikan input size konsisten
scaler = torch.amp.GradScaler("cuda")

criterion = losses.TripletMarginLoss(margin=margin)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

checkpoint_filepath = wandb_name + "_best_model.pth"
best_train_loss = float('inf')
best_epoch = 0
patience = 10
wait = 0
best_weights = None

history = {'loss': []}

process = psutil.Process(os.getpid())
psutil.cpu_percent(interval=None)
process.cpu_percent(interval=None)

print(f"Starting Embedding Training with Early Stopping (patience={patience})...")

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    
    for batch_idx, (data_eeg, targets) in enumerate(train_loader):
        if data_eeg.dim() == 3: 
            data_eeg = data_eeg.unsqueeze(1)
        data_eeg = data_eeg.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        
        optimizer.zero_grad()
        
        with torch.amp.autocast("cuda"):
            embeddings = model(data_eeg) 
            loss = criterion(embeddings, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.detach().item()
    
    avg_train_loss = train_loss / len(train_loader)
    history['loss'].append(avg_train_loss)

    memory = psutil.virtual_memory()
    process_memory = process.memory_info().rss / (1024 ** 3)

    wandb.log({
        "epoch/epoch": epoch,
        "epoch/train_loss": avg_train_loss,
        "epoch/best_train_loss": best_train_loss,
        "epoch/best_epoch": best_epoch,
        "epoch/patience": patience,
        "epoch/wait": wait,
        "resource/cpu_percent": psutil.cpu_percent(interval=None),
        "resource/process_cpu_percent": process.cpu_percent(interval=None),
        "resource/memory_percent": memory.percent,
        "resource/memory_used_gb": memory.used / (1024 ** 3),
        "resource/process_memory_gb": process_memory,
    })

    print(f"Epoch {epoch+1:03d}/{EPOCHS} | Train Loss: {avg_train_loss:.4f}")

    # Early stopping based on train loss
    if avg_train_loss < best_train_loss:
        print(f" -> Train loss improved ({best_train_loss:.4f} to {avg_train_loss:.4f}). Saving model...")
        best_train_loss = avg_train_loss
        best_epoch = epoch
        best_weights = model.state_dict().copy()
        wait = 0
        
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_train_loss': best_train_loss,
        }, checkpoint_filepath)
    else:
        wait += 1
        
    if wait >= patience:
        print(f"\nEarly stopping triggered! No improvement for {patience} epochs.")
        if best_weights is not None:
            model.load_state_dict(best_weights)
            print(f"Restored best model weights from Epoch {best_epoch+1}.")
        break

if best_weights is not None:
    model.load_state_dict(best_weights)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Training finished!")

Starting Embedding Training with Early Stopping (patience=10)...


Epoch 001/100 | Train Loss: 0.1974
 -> Train loss improved (inf to 0.1974). Saving model...


Epoch 002/100 | Train Loss: 0.1837
 -> Train loss improved (0.1974 to 0.1837). Saving model...


Epoch 003/100 | Train Loss: 0.2044


Epoch 004/100 | Train Loss: 0.1887


...

Epoch 096/100 | Train Loss: 0.0661
 -> Train loss improved (0.0692 to 0.0661). Saving model...


Epoch 097/100 | Train Loss: 0.0715


Epoch 098/100 | Train Loss: 0.0713


Epoch 099/100 | Train Loss: 0.0704


Epoch 100/100 | Train Loss: 0.0828
Training finished!


In [13]:
# Enrollment: Upload training embeddings to Qdrant
model.eval()

emb_batch = int(os.getenv("BATCH_SIZE"))
enroll_ds = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train).long(),
)
enroll_loader = DataLoader(
    enroll_ds,
    batch_size=emb_batch,
    shuffle=False,
    num_workers=int(os.getenv("NUM_WORKERS")),
    drop_last=False,
)

emb_chunks, label_chunks = [], []
with torch.no_grad():
    for data_eeg, targets in tqdm(enroll_loader, desc="Extract embeddings (enrollment)"):
        if data_eeg.dim() == 3:
            data_eeg = data_eeg.unsqueeze(1)
        data_eeg = data_eeg.to(DEVICE, non_blocking=True)
        emb = model(data_eeg).cpu().numpy()
        emb_chunks.append(emb)
        label_chunks.append(targets.numpy())

embeddings_matrix = np.concatenate(emb_chunks, axis=0)
subject_ids = np.concatenate(label_chunks, axis=0)

print(f"Embeddings shape: {embeddings_matrix.shape}")

# Upsert to Qdrant
qdrant_batch = 256
for start in tqdm(range(0, len(embeddings_matrix), qdrant_batch), desc="Upsert to Qdrant"):
    end = min(start + qdrant_batch, len(embeddings_matrix))
    points = [
        models.PointStruct(
            id=start + i,
            vector=embeddings_matrix[start + i].tolist(),
            payload={"subject_id": int(subject_ids[start + i])},
        )
        for i in range(end - start)
    ]
    client.upsert(collection_name=wandb_name, points=points)

print(f"Enrolled {len(embeddings_matrix)} embeddings to collection '{wandb_name}'.")

Extract embeddings (enrollment):   0%|                   | 0/23 [00:00<?, ?it/s]

Extract embeddings (enrollment):   4%|▍          | 1/23 [00:01<00:33,  1.51s/it]

Extract embeddings (enrollment):   9%|▉          | 2/23 [00:01<00:14,  1.46it/s]

Extract embeddings (enrollment):  13%|█▍         | 3/23 [00:01<00:08,  2.39it/s]

Extract embeddings (enrollment):  17%|█▉         | 4/23 [00:01<00:05,  3.41it/s]

...

Upsert to Qdrant:  50%|█████████████             | 6/12 [00:00<00:00, 11.44it/s]

Upsert to Qdrant:  67%|█████████████████▎        | 8/12 [00:00<00:00, 12.72it/s]

Upsert to Qdrant:  92%|██████████████████████▉  | 11/12 [00:00<00:00, 14.96it/s]

Upsert to Qdrant: 100%|█████████████████████████| 12/12 [00:00<00:00, 12.35it/s]

Enrolled 2943 embeddings to collection 'embedding_v4_ec_train_90_2024_2_2_b128_e100_margin_0.2'.


# model = torch.load('../best_eeg_embedding_model.pth') <br>
coba modifikasi arsitektur model (conv 1d -> 2d) <br>
perkecil kernel size (8 -> ...) <br>
cek performance per satu subject (waktu, accuracy) <br>
cek usage cpu + memory <br>

In [14]:
# Euclidean distance evaluation - Run multiple times for stability check
# Note: For Euclidean, LOWER score = MORE similar (opposite of cosine)

NUM_EVAL_RUNS = 1
roc_query_limit = 200

eval_results = {
    'top1_accuracy': [],
    'eer': [],
    'eer_threshold': [],
    'eval_time': [],
    'genuine_count': [],
    'impostor_count': []
}

model.eval()

for run_idx in range(NUM_EVAL_RUNS):
    y_true = []
    y_scores = []
    top1_correct = 0
    total_test_samples = 0
    
    run_start = time.time()
    
    with torch.no_grad():
        for data_eeg, targets in test_loader:
            data_eeg = data_eeg.to("cuda", non_blocking=True)
            embeddings = model(data_eeg).cpu().numpy()
            targets = targets.numpy()

            for i in range(len(embeddings)):
                query_vector = embeddings[i].tolist()
                true_label = int(targets[i])

                search_result = client.query_points(
                    collection_name=wandb_name,
                    query=query_vector,
                    limit=roc_query_limit
                )
                points = search_result.points
                if not points:
                    continue

                best_match = points[0]
                predicted_label = int(best_match.payload["subject_id"])
                top1_correct += int(predicted_label == true_label)
                total_test_samples += 1

                for point in points:
                    candidate_label = int(point.payload["subject_id"])
                    y_true.append(1 if candidate_label == true_label else 0)
                    y_scores.append(-point.score)

    y_true = np.array(y_true)
    y_scores = np.array(y_scores)

    classes, counts = np.unique(y_true, return_counts=True)
    class_counts = dict(zip(classes.tolist(), counts.tolist()))
    
    fpr, tpr, thresholds = roc_curve(y_true, y_scores)
    far = fpr
    frr = 1 - tpr

    eer_idx = np.nanargmin(np.abs(far - frr))
    eer = (far[eer_idx] + frr[eer_idx]) / 2
    eer_thresh = -thresholds[eer_idx]
    top1_acc = top1_correct / total_test_samples
    run_time = time.time() - run_start
    
    eval_results['top1_accuracy'].append(top1_acc)
    eval_results['eer'].append(eer)
    eval_results['eer_threshold'].append(eer_thresh)
    eval_results['eval_time'].append(run_time)
    eval_results['genuine_count'].append(class_counts.get(1, 0))
    eval_results['impostor_count'].append(class_counts.get(0, 0))
    
    print(f"Run {run_idx+1:02d}/{NUM_EVAL_RUNS} | Top-1: {top1_acc*100:.2f}% | EER: {eer*100:.2f}% | Time: {run_time:.2f}s")

# Calculate statistics
top1_mean = np.mean(eval_results['top1_accuracy']) * 100
top1_std = np.std(eval_results['top1_accuracy']) * 100
eer_mean = np.mean(eval_results['eer']) * 100
eer_std = np.std(eval_results['eer']) * 100
thresh_mean = np.mean(eval_results['eer_threshold'])
thresh_std = np.std(eval_results['eer_threshold'])
time_mean = np.mean(eval_results['eval_time'])
genuine_total = int(np.mean(eval_results['genuine_count']))
impostor_total = int(np.mean(eval_results['impostor_count']))

print("\n" + "="*60)
print("=== STABILITY EVALUATION RESULTS (20 RUNS) ===")
print("="*60)
print(f"Top-1 Accuracy : {top1_mean:.2f}% ± {top1_std:.4f}%")
print(f"EER            : {eer_mean:.2f}% ± {eer_std:.4f}%")
print(f"EER Threshold  : {thresh_mean:.4f} ± {thresh_std:.6f}")
print(f"Genuine/Impostor: {genuine_total} / {impostor_total}")
print(f"Avg Eval Time  : {time_mean:.2f}s")
print("="*60)

# Use mean values for subsequent cells
eer_threshold = thresh_mean
top1_accuracy = top1_mean / 100
eer = eer_mean / 100

# Log to wandb
wandb.log({
    "stability/top1_mean": top1_mean,
    "stability/top1_std": top1_std,
    "stability/eer_mean": eer_mean,
    "stability/eer_std": eer_std,
    "stability/threshold_mean": thresh_mean,
    "stability/threshold_std": thresh_std,
    "stability/genuine_count": genuine_total,
    "stability/impostor_count": impostor_total,
    "stability/num_runs": NUM_EVAL_RUNS,
})

Run 01/1 | Top-1: 97.86% | EER: 7.60% | Time: 2.52s

=== STABILITY EVALUATION RESULTS (20 RUNS) ===
Top-1 Accuracy : 97.86% ± 0.0000%
EER            : 7.60% ± 0.0000%
EER Threshold  : 0.6072 ± 0.000000
Genuine/Impostor: 8814 / 56586
Avg Eval Time  : 2.52s


In [15]:
# Single-subject verification with genuine + impostor testing
# CORRECT APPROACH: Query only the CLAIMED identity's enrolled samples
# Verification asks: "Is this person who they claim to be (subject X)?"

target_subject_id = 0
num_impostors_to_test = 5

print(f"=== VERIFICATION TEST FOR SUBJECT {target_subject_id} ===\n")

# --- GENUINE ATTEMPTS (should be accepted) ---
subject_mask = y_test == target_subject_id
X_genuine = X_test[subject_mask]
y_genuine = y_test[subject_mask]

print(f"Genuine samples: {len(X_genuine)}")

genuine_ds = TensorDataset(
    torch.from_numpy(X_genuine).float(),
    torch.from_numpy(y_genuine).long(),
)
genuine_loader = DataLoader(
    genuine_ds,
    batch_size=int(os.getenv("BATCH_SIZE")),
    num_workers=int(os.getenv("NUM_WORKERS")),
    drop_last=False,
)

# --- IMPOSTOR ATTEMPTS (should be rejected) ---
other_subjects = [s for s in np.unique(y_test) if s != target_subject_id]
impostor_subjects = np.random.choice(other_subjects, size=min(num_impostors_to_test, len(other_subjects)), replace=False)

impostor_mask = np.isin(y_test, impostor_subjects)
X_impostor = X_test[impostor_mask]
y_impostor = y_test[impostor_mask]

print(f"Impostor samples: {len(X_impostor)} (from subjects {impostor_subjects.tolist()})")

impostor_ds = TensorDataset(
    torch.from_numpy(X_impostor).float(),
    torch.from_numpy(y_impostor).long(),
)
impostor_loader = DataLoader(
    impostor_ds,
    batch_size=int(os.getenv("BATCH_SIZE")),
    num_workers=int(os.getenv("NUM_WORKERS")),
    drop_last=False,
)

def compute_exact_euclidean(query_emb, candidate_embs):
    return np.sqrt(np.sum((candidate_embs - query_emb) ** 2, axis=1))

def verify_against_claimed_identity(loader, is_genuine, model, client, collection_name, threshold, claimed_id):
    correct = 0
    total = 0
    distances = []
    details = []

    model.eval()
    with torch.no_grad():
        for data_eeg, targets in loader:
            data_eeg = data_eeg.to("cuda", non_blocking=True)
            embeddings = model(data_eeg).cpu().numpy()

            for i in range(len(embeddings)):
                query_vector = embeddings[i]

                result = client.query_points(
                    collection_name=collection_name,
                    query=query_vector.tolist(),
                    query_filter=models.Filter(
                        must=[models.FieldCondition(
                            key="subject_id",
                            match=models.MatchValue(value=claimed_id)
                        )]
                    ),
                    limit=10,
                    with_vectors=True
                )

                if not result.points:
                    continue

                candidate_vectors = np.array([p.vector for p in result.points])
                exact_distances = compute_exact_euclidean(query_vector, candidate_vectors)
                min_distance = np.min(exact_distances)

                is_accepted = (min_distance <= threshold)

                if is_genuine:
                    correct_decision = is_accepted
                else:
                    correct_decision = not is_accepted

                correct += int(correct_decision)
                total += 1
                distances.append(min_distance)
                details.append({
                    'is_genuine': is_genuine,
                    'distance_to_claimed': min_distance,
                    'accepted': is_accepted,
                    'correct': correct_decision
                })

    return correct, total, distances, details

# Run verification
eval_start_time = time.time()

genuine_correct, genuine_total, genuine_distances, genuine_details = verify_against_claimed_identity(
    genuine_loader, is_genuine=True, model=model, client=client,
    collection_name=wandb_name, threshold=eer_threshold, claimed_id=target_subject_id
)

impostor_correct, impostor_total, impostor_distances, impostor_details = verify_against_claimed_identity(
    impostor_loader, is_genuine=False, model=model, client=client,
    collection_name=wandb_name, threshold=eer_threshold, claimed_id=target_subject_id
)

eval_elapsed_time = time.time() - eval_start_time

# Calculate metrics
tar = genuine_correct / genuine_total  # True Accept Rate
frr = 1 - tar  # False Reject Rate
trr = impostor_correct / impostor_total  # True Reject Rate
far = 1 - trr  # False Accept Rate

print("\n" + "="*60)
print(f"=== VERIFICATION RESULTS (Subject {target_subject_id}) ===")
print("="*60)
print(f"\nGENUINE ATTEMPTS (claiming to be subject {target_subject_id}):")
print(f"  Samples tested  : {genuine_total}")
print(f"  TAR (accepted)  : {tar * 100:.2f}%")
print(f"  FRR (rejected)  : {frr * 100:.2f}%")
print(f"  Mean distance   : {np.mean(genuine_distances):.4f}")
print(f"  Min/Max distance: {np.min(genuine_distances):.4f} / {np.max(genuine_distances):.4f}")

print(f"\nIMPOSTOR ATTEMPTS (falsely claiming to be subject {target_subject_id}):")
print(f"  Samples tested  : {impostor_total}")
print(f"  TRR (rejected)  : {trr * 100:.2f}%")
print(f"  FAR (accepted)  : {far * 100:.2f}%")
print(f"  Mean distance   : {np.mean(impostor_distances):.4f}")
print(f"  Min/Max distance: {np.min(impostor_distances):.4f} / {np.max(impostor_distances):.4f}")

print(f"\nOVERALL:")
print(f"  EER Threshold   : {eer_threshold:.4f}")
print(f"  Time taken      : {eval_elapsed_time:.2f}s")
print("="*60)

# Log verification metrics to wandb
wandb.log({
    "verification/target_subject": target_subject_id,
    "verification/tar": tar * 100,
    "verification/frr": frr * 100,
    "verification/trr": trr * 100,
    "verification/far": far * 100,
    "verification/genuine_mean_distance": np.mean(genuine_distances),
    "verification/genuine_min_distance": np.min(genuine_distances),
    "verification/genuine_max_distance": np.max(genuine_distances),
    "verification/impostor_mean_distance": np.mean(impostor_distances),
    "verification/impostor_min_distance": np.min(impostor_distances),
    "verification/impostor_max_distance": np.max(impostor_distances),
    "verification/threshold": eer_threshold,
    "verification/time_taken": eval_elapsed_time,
    "verification/genuine_samples": genuine_total,
    "verification/impostor_samples": impostor_total,
})

# Finish wandb run
wandb.finish()

=== VERIFICATION TEST FOR SUBJECT 0 ===

Genuine samples: 3
Impostor samples: 15 (from subjects [99, 38, 17, 72, 16])


wandb: 
wandb: Run history:
wandb:        epoch/best_epoch ▁▁▁▁▁▂▂▂▂▂▂▂▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇█
wandb:   epoch/best_train_loss  ██▇▇▇▇▆▆▆▆▆▅▅▅▅▅▅▅▄▄▄▄▄▄▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁
wandb:             epoch/epoch ▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▇▇▇▇▇▇▇███
wandb:          epoch/patience ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:        epoch/train_loss █▇█▇▇▇▆▆▆▆▆▅▆▆▅▅▅▅▅▅▄▄▄▄▃▃▄▃▃▃▃▂▂▂▂▂▁▂▁▂
wandb:              epoch/wait ▁▁▂▅▁▃▅▆▅▁▁▂▂▃▁▂▃▁▂▃▁▂▃▅▁▆▂▁▆▂▁▃▆▁█▂▃▆▁▂
wandb:    resource/cpu_percent ▁▇▆█▇▇▇▇██▆▆▆▆▆▇▆▅▅▆▆▆▇▇▇▅▆▆▆▇▇▆▆▆▇▆▅▇▆▅
wandb:  resource/logging_check ▁
wandb: resource/memory_percent ▁▂▁▄▁▃▄▂▃▃▂▄▃▄▅▇▄▃▂▆▄▄▄▅▇██▅▅▅▄▅▄█▆▇█▇█▅
wandb: resource/memory_used_gb ▂▃▃▃▂▃▃▁▄▄▅▅▇▄▄▅▄▅▇▇▄▄▄▄▇▆▆█▆▅▅▆▄▇▆▇▇▇▇▆
wandb:                     +26 ...
wandb: 
wandb: Run summary:
wandb:        epoch/best_epoch 95
wandb:   epoch/best_train_loss 0.06606
wandb:             epoch/epoch 99
wandb:          epoch/patience 10
wandb:        epoch/train_loss 0.08281
wandb:              epoch/wait 3
wandb:    resou

wandb: 🚀 View run embedding_v4_ec_train_90_2024_2_2_b128_e100_margin_0.2 at: http://localhost:8080/chocomaltt/eeg-biometric-system/runs/o970vor3
wandb: ⭐️ View project at: http://localhost:8080/chocomaltt/eeg-biometric-system
wandb: Synced 7 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20260531_132136-o970vor3/logs



=== VERIFICATION RESULTS (Subject 0) ===

GENUINE ATTEMPTS (claiming to be subject 0):
  Samples tested  : 3
  TAR (accepted)  : 100.00%
  FRR (rejected)  : 0.00%
  Mean distance   : 0.2743
  Min/Max distance: 0.2643 / 0.2942

IMPOSTOR ATTEMPTS (falsely claiming to be subject 0):
  Samples tested  : 15
  TRR (rejected)  : 100.00%
  FAR (accepted)  : 0.00%
  Mean distance   : 1.1402
  Min/Max distance: 0.9104 / 1.3286

OVERALL:
  EER Threshold   : 0.6072
  Time taken      : 1.08s


In [16]:
os.makedirs("models", exist_ok=True)
torch.save(model, "models/" + wandb_name + ".pth")

## Hasil Bimbingan 
1. Filtering belum ada (DONE)
2. Windowing pakai beberapa scenario (win_size=1, stride=1, win_size=2, stride=1, win_size=1, stride=2)
3. coba eNN (Euclidean Distance)
4. visualisasi data di qdrant
5. Dokumentasi waktu testing
6. Bikin set data splitting dengan seeder berbeda (min. 10)